<a href="https://colab.research.google.com/github/ishaqafridi/Disease-specific-KG/blob/main/Knowledge_Graph__(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Drug-Disease Knowledge Graph: A Lightweight Alternative to PrimeKG

This notebook builds a biomedical knowledge graph from **two data repositories**:
1. `pharma_1.csv` - Drug names, classes, indications, side effects, manufacturers
2. `Diseases_Symptoms.csv` - Disease-symptom associations

The graph contains **5 node types** (Drugs, Diseases, Side Effects, Drug Classes, Manufacturers) and **6 relationship types** including TREATS, HAS_SIDE_EFFECT, and inferred CONTRAINDICATED_FOR edges. Five interactive visualizations (network, sunburst, sankey, heatmap, ego network) enable exploration of drug-disease relationships. This represents an **alternative to PrimeKG** using rule-based inference instead of multiple database integration.

### BLOCK 1: IMPORTS AND SETUP

In [ ]:
# Cell 1: Install required packages (run once)
!pip install networkx plotly python-louvain pandas numpy -q

# Cell 2: Import libraries
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import plotly.graph_objects as go
import plotly.express as px
import json
import warnings
warnings.filterwarnings('ignore')

print("✓ Setup complete")

✓ Setup complete


### BLOCK 2: LOAD BOTH DATA RESOURCES

In [ ]:
# Cell 3: Load BOTH data repositories
print("=" * 60)
print("LOADING TWO DATA REPOSITORIES")
print("=" * 60)

# REPOSITORY 1: Pharmaceutical data
df_pharma = pd.read_csv('pharma_1.csv')
print(f"\n📦 REPOSITORY 1: pharma_1.csv")
print(f"   - Shape: {df_pharma.shape}")
print(f"   - Columns: {list(df_pharma.columns)}")

# REPOSITORY 2: Disease-Symptoms data
df_disease = pd.read_csv('Diseases_Symptoms.csv')
print(f"\n📦 REPOSITORY 2: Diseases_Symptoms.csv")
print(f"   - Shape: {df_disease.shape}")
print(f"   - Columns: {list(df_disease.columns)}")

# Clean both datasets
df_pharma['Side Effects'] = df_pharma['Side Effects'].fillna('')
df_pharma['Dosage Strength'] = df_pharma['Dosage Strength'].fillna('Unknown')
df_pharma['Dosage Form'] = df_pharma['Dosage Form'].fillna('Unknown')

print("\n✓ Both datasets loaded and cleaned")

LOADING TWO DATA REPOSITORIES

📦 REPOSITORY 1: pharma_1.csv
   - Shape: (20000, 12)
   - Columns: ['Drug Name', 'Drug Class', 'Indication', 'Side Effects', 'Dosage Form', 'Dosage Strength', 'Prescription Status', 'Manufacturer', 'Approval Year', 'Description', 'Price (USD)', 'Market Status']

📦 REPOSITORY 2: Diseases_Symptoms.csv
   - Shape: (405, 6)
   - Columns: ['Name', 'Symptoms', 'Treatments', 'Disease_Code', 'Contagious', 'Chronic']

✓ Both datasets loaded and cleaned


In [ ]:
# Cell 4: Quick preview of both datasets
print("First 3 rows of pharma_1.csv:")
df_pharma.head(3)

print("\nFirst 3 rows of Diseases_Symptoms.csv:")
df_disease.head(3)

First 3 rows of pharma_1.csv:

First 3 rows of Diseases_Symptoms.csv:


,Name,Symptoms,Treatments,Disease_Code,Contagious,Chronic
0,Gestational Cholestasis,"Itchy skin, particularly on the hands and feet",NaN,D001,False,False
1,Injury to Internal Organ,"Abdominal pain, bleeding, organ dysfunction","Immediate medical attention, diagnostic tests,...",D002,False,False
2,Scabies,"Intense itching, especially at night, small bl...",Prescription medications (topical or oral scab...,D003,False,True


### BLOCK 3: KNOWLEDGE GRAPH CLASS

In [ ]:
# Cell 5: Define the Knowledge Graph class (with get_nodes_by_type method)
class KnowledgeGraph:
    def __init__(self):
        self.nodes = {}      # node_id -> {'type': str, 'attributes': dict}
        self.edges = []      # list of {'source', 'target', 'relation', 'attributes'}

    def add_node(self, node_id, node_type, **attrs):
        if node_id not in self.nodes:
            self.nodes[node_id] = {'type': node_type, 'attributes': attrs}
        else:
            self.nodes[node_id]['attributes'].update(attrs)

    def add_edge(self, source, target, relation, **attrs):
        if source not in self.nodes:
            self.add_node(source, 'Unknown')
        if target not in self.nodes:
            self.add_node(target, 'Unknown')
        self.edges.append({'source': source, 'target': target,
                          'relation': relation, 'attributes': attrs})

    def get_nodes_by_type(self, node_type):
        """Return list of node IDs with given type"""
        return [nid for nid, data in self.nodes.items()
                if data['type'] == node_type]

    def get_edges_by_relation(self, relation):
        """Return list of edges with given relation"""
        return [e for e in self.edges if e['relation'] == relation]

    def to_networkx(self):
        G = nx.MultiDiGraph()
        for nid, data in self.nodes.items():
            G.add_node(nid, type=data['type'], **data['attributes'])
        for e in self.edges:
            G.add_edge(e['source'], e['target'], relation=e['relation'])
        return G

    def summary(self):
        print(f"Nodes: {len(self.nodes)} | Edges: {len(self.edges)}")
        types = Counter([d['type'] for d in self.nodes.values()])
        for t, c in types.most_common():
            print(f"  {t}: {c}")

kg = KnowledgeGraph()
print("✓ KnowledgeGraph class ready")

✓ KnowledgeGraph class ready


### BLOCK 4: BUILD GRAPH FROM REPOSITORY 1 (pharma_1.csv)

In [ ]:
# Cell 6: Extract nodes and edges from pharma_1.csv
print("=" * 60)
print("BUILDING GRAPH FROM REPOSITORY 1: pharma_1.csv")
print("=" * 60)

# Add Drug nodes
for _, row in df_pharma.iterrows():
    kg.add_node(row['Drug Name'], 'Drug',
                drug_class=row['Drug Class'],
                dosage_strength=row['Dosage Strength'],
                dosage_form=row['Dosage Form'],
                prescription_status=row['Prescription Status'],
                manufacturer=row['Manufacturer'],
                price_usd=row['Price (USD)'],
                market_status=row['Market Status'])

print(f"✓ Added {len(kg.get_nodes_by_type('Drug'))} Drugs")

# Add Disease nodes and TREATS edges
for _, row in df_pharma.iterrows():
    indication = row['Indication']
    if pd.notna(indication) and indication:
        kg.add_node(indication, 'Disease')
        kg.add_edge(row['Drug Name'], indication, 'TREATS')

print(f"✓ Added {len(kg.get_nodes_by_type('Disease'))} Diseases")
print(f"✓ Added {len(kg.get_edges_by_relation('TREATS'))} TREATS edges")

BUILDING GRAPH FROM REPOSITORY 1: pharma_1.csv
✓ Added 8031 Drugs
✓ Added 10 Diseases
✓ Added 20000 TREATS edges


In [ ]:
# Cell 7: Add Side Effects, Drug Classes, Manufacturers
# Add Side Effect nodes and HAS_SIDE_EFFECT edges
for _, row in df_pharma.iterrows():
    if row['Side Effects'] and pd.notna(row['Side Effects']):
        for effect in str(row['Side Effects']).split(','):
            effect = effect.strip()
            if effect and effect != 'nan':
                kg.add_node(effect, 'SideEffect')
                kg.add_edge(row['Drug Name'], effect, 'HAS_SIDE_EFFECT')

print(f"✓ Added {len(kg.get_nodes_by_type('SideEffect'))} Side Effects")

# Add Drug Class nodes and HAS_CLASS edges
for _, row in df_pharma.iterrows():
    drug_class = row['Drug Class']
    if pd.notna(drug_class) and drug_class:
        kg.add_node(drug_class, 'DrugClass')
        kg.add_edge(row['Drug Name'], drug_class, 'HAS_CLASS')

print(f"✓ Added {len(kg.get_nodes_by_type('DrugClass'))} Drug Classes")

# Add Manufacturer nodes and MANUFACTURED_BY edges
for _, row in df_pharma.iterrows():
    mfr = row['Manufacturer']
    if pd.notna(mfr) and mfr:
        kg.add_node(mfr, 'Manufacturer')
        kg.add_edge(row['Drug Name'], mfr, 'MANUFACTURED_BY')

print(f"✓ Added {len(kg.get_nodes_by_type('Manufacturer'))} Manufacturers")

print("\n📊 GRAPH SUMMARY AFTER REPOSITORY 1:")
kg.summary()

✓ Added 6 Side Effects
✓ Added 10 Drug Classes
✓ Added 5 Manufacturers

📊 GRAPH SUMMARY AFTER REPOSITORY 1:
Nodes: 8062 | Edges: 99997
  Drug: 8031
  Disease: 10
  DrugClass: 10
  SideEffect: 6
  Manufacturer: 5


### BLOCK 5: INTEGRATE REPOSITORY 2 (Diseases_Symptoms.csv)

In [ ]:
# Cell 8: Build symptom-disease map from Diseases_Symptoms.csv
print("=" * 60)
print("INTEGRATING REPOSITORY 2: Diseases_Symptoms.csv")
print("=" * 60)

# Build mapping from Diseases_Symptoms.csv
symptom_to_diseases = defaultdict(set)

# Detect column structure
disease_col = None
symptom_cols = []

for col in df_disease.columns:
    if 'disease' in col.lower() or 'condition' in col.lower() or 'label' in col.lower():
        disease_col = col
    elif 'symptom' in col.lower():
        symptom_cols.append(col)

# If no symptom columns found, use all columns except disease column
if not symptom_cols and disease_col:
    symptom_cols = [col for col in df_disease.columns if col != disease_col]

print(f"Detected Disease column: {disease_col}")
print(f"Detected Symptom columns: {symptom_cols[:5]}...")

# Build the mapping
for _, row in df_disease.iterrows():
    if disease_col and pd.notna(row[disease_col]):
        disease = str(row[disease_col]).strip()
        for symp_col in symptom_cols:
            symptom = row[symp_col]
            if pd.notna(symptom) and str(symptom).strip():
                symptom_to_diseases[str(symptom).strip()].add(disease)

# Save as JSON (second repository artifact)
with open('symptom_disease_map.json', 'w') as f:
    json.dump({k: list(v) for k, v in symptom_to_diseases.items()}, f, indent=2)

print(f"\n✓ Built symptom-disease map from Diseases_Symptoms.csv")
print(f"   - Unique symptoms: {len(symptom_to_diseases)}")
print(f"   - Unique diseases: {len(set(d for v in symptom_to_diseases.values() for d in v))}")
print(f"\nSample mappings:")
for s, ds in list(symptom_to_diseases.items())[:5]:
    print(f"  '{s}' → {list(ds)[:3]}")

INTEGRATING REPOSITORY 2: Diseases_Symptoms.csv
Detected Disease column: Disease_Code
Detected Symptom columns: ['Symptoms']...

✓ Built symptom-disease map from Diseases_Symptoms.csv
   - Unique symptoms: 400
   - Unique diseases: 395

Sample mappings:
  'Itchy skin, particularly on the hands and feet' → ['D001']
  'Abdominal pain, bleeding, organ dysfunction' → ['D002']
  'Intense itching, especially at night, small blisters or bumps' → ['D003']
  'Cloudy or hazy eyes, excessive tearing, sensitivity to light' → ['D004']
  'Avoidance or restriction of certain foods or entire food groups, significant weight loss or failure to gain weight' → ['D005']


In [ ]:
# Cell 9: Enriched inference using BOTH repositories (FIXED)
print("=" * 60)
print("ENRICHED INFERENCE: Repository 1 + Repository 2")
print("=" * 60)

# Get data from Repository 1 (pharma_1.csv)
drug_to_diseases_primary = defaultdict(set)
drug_to_symptoms_primary = defaultdict(set)

for e in kg.edges:
    if e['relation'] == 'TREATS':
        drug_to_diseases_primary[e['source']].add(e['target'])
    elif e['relation'] == 'HAS_SIDE_EFFECT':
        drug_to_symptoms_primary[e['source']].add(e['target'])

print(f"\nDrug side effects: {len(drug_to_symptoms_primary)} drugs with symptoms")

# Build keyword mapping from clinical symptoms to simple symptoms
print("\nBuilding symptom keyword mapping...")

# Map simple drug symptoms to keywords for matching
simple_symptom_keywords = {
    'Headache': ['headache', 'migraine', 'cephalalgia', 'head pain', 'head ache'],
    'Nausea': ['nausea', 'vomiting', 'emesis', 'queasy', 'sick stomach'],
    'Stomach Upset': ['stomach', 'abdominal', 'gastrointestinal', 'indigestion', 'upset stomach', 'belly'],
    'Fatigue': ['fatigue', 'tired', 'exhaustion', 'lethargy', 'weakness', 'low energy'],
    'Drowsiness': ['drowsy', 'sleepy', 'somnolence', 'drowsiness', 'drows', 'sleepiness'],
    'Dizziness': ['dizzy', 'vertigo', 'lightheaded', 'dizziness', 'light-headed']
}

# Create mapping from simple symptom to diseases
simple_to_diseases = defaultdict(set)

for simple_symptom, keywords in simple_symptom_keywords.items():
    for clinical_symptom, diseases in symptom_to_diseases.items():
        clinical_lower = clinical_symptom.lower()
        for keyword in keywords:
            if keyword in clinical_lower:
                simple_to_diseases[simple_symptom].update(diseases)
                break
        if simple_symptom.lower() in clinical_lower:
            simple_to_diseases[simple_symptom].update(diseases)

print(f"  Mapped {len(simple_to_diseases)} simple symptoms to diseases")

# Also add direct symptom name matching (case-insensitive)
for clinical_symptom, diseases in symptom_to_diseases.items():
    for simple_symptom in simple_symptom_keywords.keys():
        if simple_symptom.lower() in clinical_symptom.lower():
            simple_to_diseases[simple_symptom].update(diseases)

# Infer new contraindications
contraindications_external = []
matched_count = 0

for drug, symptoms in drug_to_symptoms_primary.items():
    treats_diseases = drug_to_diseases_primary.get(drug, set())

    for symptom in symptoms:
        if symptom in simple_to_diseases:
            for disease in simple_to_diseases[symptom]:
                if disease not in treats_diseases:
                    # Check if edge already exists
                    edge_exists = any(e['source'] == drug and e['target'] == disease and
                                     e['relation'] == 'CONTRAINDICATED_FROM_EXTERNAL'
                                     for e in kg.edges)
                    if not edge_exists:
                        kg.add_edge(drug, disease, 'CONTRAINDICATED_FROM_EXTERNAL',
                                   confidence='high',
                                   source_repo='Diseases_Symptoms.csv',
                                   inference_rule=f"{drug} causes {symptom}, which indicates {disease}")
                        contraindications_external.append((drug, disease, symptom))
                        matched_count += 1

                        # Limit to 100 total contraindications (enough for visualization)
                        if matched_count >= 100:
                            break
            if matched_count >= 100:
                break
    if matched_count >= 100:
        break

print(f"\n✓ Added {len(contraindications_external)} new contraindications from Repository 2!")

if contraindications_external:
    print("\nSample enriched contraindications:")
    for drug, disease, symptom in contraindications_external[:10]:
        print(f"  💊 {drug[:20]} → ⚠️ {disease[:25]} (via: {symptom})")

    # Show which symptoms matched
    matched_symptoms = set([s for _, _, s in contraindications_external])
    print(f"\n  Matched symptoms: {matched_symptoms}")
else:
    print("\n⚠️ No matches found. Adding demo contraindications for visualization...")

    # Add demo contraindications so visualization shows something
    demo_drugs = list(drug_to_symptoms_primary.keys())[:5]
    demo_diseases = ['D001', 'D002', 'D003', 'D004', 'D005']

    for i, drug in enumerate(demo_drugs):
        for j, disease in enumerate(demo_diseases):
            if i != j and len(contraindications_external) < 20:
                kg.add_edge(drug, disease, 'CONTRAINDICATED_FROM_EXTERNAL',
                           confidence='demo',
                           source_repo='demo_data',
                           inference_rule=f"Demo contraindication for testing")
                contraindications_external.append((drug, disease, "demo_symptom"))

    print(f"  Added {len(contraindications_external)} demo contraindications")

# Show final count
contra_count = len(kg.get_edges_by_relation('CONTRAINDICATED_FROM_EXTERNAL'))
print(f"\n✅ Total CONTRAINDICATED_FROM_EXTERNAL edges in KG: {contra_count}")

ENRICHED INFERENCE: Repository 1 + Repository 2

Drug side effects: 8031 drugs with symptoms

Building symptom keyword mapping...
  Mapped 6 simple symptoms to diseases

✓ Added 100 new contraindications from Repository 2!

Sample enriched contraindications:
  💊 Drug-7690 → ⚠️ D178 (via: Drowsiness)
  💊 Drug-7690 → ⚠️ D382 (via: Drowsiness)
  💊 Drug-7690 → ⚠️ D361 (via: Drowsiness)
  💊 Drug-7690 → ⚠️ D124 (via: Drowsiness)
  💊 Drug-7690 → ⚠️ D007 (via: Drowsiness)
  💊 Drug-7690 → ⚠️ D368 (via: Drowsiness)
  💊 Drug-7690 → ⚠️ D123 (via: Drowsiness)
  💊 Drug-7690 → ⚠️ D045 (via: Nausea)
  💊 Drug-7690 → ⚠️ D258 (via: Nausea)
  💊 Drug-7690 → ⚠️ D101 (via: Nausea)

  Matched symptoms: {'Drowsiness', 'Nausea', 'Stomach Upset', 'Headache', 'Fatigue'}

✅ Total CONTRAINDICATED_FROM_EXTERNAL edges in KG: 100


### BLOCK 6: CONVERT TO NETWORKX

In [ ]:
# Cell 10: Convert to NetworkX for analysis
G = kg.to_networkx()

print(f"NetworkX graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Graph density: {nx.density(G):.4f}")

# Find hub nodes (most connected)
degrees = dict(G.degree())
top_hubs = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop 10 hub nodes:")
for node, deg in top_hubs:
    node_type = kg.nodes[node]['type'] if node in kg.nodes else 'Unknown'
    print(f"  {node[:30]} ({node_type}): degree={deg}")

NetworkX graph: 8162 nodes, 100097 edges
Graph density: 0.0015

Top 10 hub nodes:
  Fatigue (SideEffect): degree=6766
  Nausea (SideEffect): degree=6717
  Drowsiness (SideEffect): degree=6642
  Stomach Upset (SideEffect): degree=6640
  Headache (SideEffect): degree=6617
  Dizziness (SideEffect): degree=6615
  MediLife (Manufacturer): degree=4110
  PharmaCorp (Manufacturer): degree=3996
  HealthPlus (Manufacturer): degree=3987
  Global Remedies (Manufacturer): degree=3957


### BLOCK 7: INTERACTIVE NETWORK VISUALIZATION

In [ ]:
# Cell 11: Create interactive network visualization (FIXED)
def plot_network(kg, G, max_nodes=60):
    degrees = dict(G.degree())
    top_nodes = sorted(degrees, key=degrees.get, reverse=True)[:max_nodes]
    G_sub = G.subgraph(top_nodes)

    pos = nx.spring_layout(G_sub, k=2, seed=42)

    colors = {'TREATS': '#2ecc71', 'HAS_SIDE_EFFECT': '#e74c3c',
              'HAS_CLASS': '#3498db', 'MANUFACTURED_BY': '#f39c12',
              'CONTRAINDICATED_FOR': '#9b59b6', 'CONTRAINDICATED_FROM_EXTERNAL': '#1abc9c'}

    edge_traces = []
    for rel, color in colors.items():
        edges = [(u,v) for u,v,k,d in G_sub.edges(keys=True, data=True)
                 if d.get('relation') == rel]
        if edges:
            x_vals, y_vals = [], []
            for u,v in edges:
                x_vals += [pos[u][0], pos[v][0], None]
                y_vals += [pos[u][1], pos[v][1], None]
            edge_traces.append(go.Scatter(x=x_vals, y=y_vals, mode='lines',
                                          line=dict(width=1, color=color),
                                          name=rel, hoverinfo='none'))

    node_x, node_y, node_text, node_colors, node_sizes = [], [], [], [], []
    color_map = {'Drug': '#ff6b6b', 'Disease': '#4ecdc4', 'SideEffect': '#ffe66d',
                 'DrugClass': '#95e77e', 'Manufacturer': '#ff9f1c'}

    for node in G_sub.nodes():
        x, y = pos[node]
        node_x.append(x); node_y.append(y)
        node_type = kg.nodes.get(node, {}).get('type', 'Unknown')
        node_colors.append(color_map.get(node_type, '#95a5a6'))
        node_text.append(f"<b>{node}</b><br>Type: {node_type}")
        node_sizes.append(12 + G_sub.degree(node))

    node_trace = go.Scatter(x=node_x, y=node_y, mode='markers+text',
                            text=[n[:15] for n in G_sub.nodes()],
                            textposition="top center", hoverinfo='text',
                            hovertext=node_text,
                            marker=dict(size=node_sizes, color=node_colors,
                                       line=dict(width=1, color='white')))

    fig = go.Figure(data=edge_traces + [node_trace],
                    layout=go.Layout(title='Drug-Disease Knowledge Graph (Two Repositories)',
                                     showlegend=True, hovermode='closest',
                                     width=1400, height=1000,
                                     xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                                     yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                                     plot_bgcolor='rgba(0,0,0,0)'))
    return fig

fig = plot_network(kg, G)
fig.show()

### BLOCK 8: SUNBURST CHART

In [ ]:
# Cell 12: Sunburst chart - Drug class hierarchy
labels, parents, values = ['All Drugs'], [''], [len(kg.get_nodes_by_type('Drug'))]

class_stats = {}
for drug in kg.get_nodes_by_type('Drug'):
    dc = kg.nodes[drug]['attributes'].get('drug_class', 'Unknown')
    if dc not in class_stats:
        class_stats[dc] = {'drugs': [], 'indications': set(), 'side_effects': set()}
    class_stats[dc]['drugs'].append(drug)
    for e in kg.edges:
        if e['source'] == drug and e['relation'] == 'TREATS':
            class_stats[dc]['indications'].add(e['target'])
        elif e['source'] == drug and e['relation'] == 'HAS_SIDE_EFFECT':
            class_stats[dc]['side_effects'].add(e['target'])

for dc, stats in class_stats.items():
    labels.append(dc); parents.append('All Drugs'); values.append(len(stats['drugs']))
    if stats['indications']:
        labels.append(f"{dc} - Indications"); parents.append(dc); values.append(len(stats['indications']))
    if stats['side_effects']:
        labels.append(f"{dc} - Side Effects"); parents.append(dc); values.append(len(stats['side_effects']))

fig = go.Figure(go.Sunburst(labels=labels, parents=parents, values=values,
                            branchvalues='total', marker=dict(colorscale='RdBu'),
                            textinfo='label+percent entry'))
fig.update_layout(title='Drug Class Hierarchy: Indications vs Side Effects', width=800, height=800)
fig.show()

### BLOCK 9: SANKEY DIAGRAM

In [ ]:
# Cell 13: Sankey diagram - Drug to Disease to Side Effect flows
drug_degrees = [(drug, G.degree(drug)) for drug in kg.get_nodes_by_type('Drug')]
top_drugs = [d for d, deg in sorted(drug_degrees, key=lambda x: x[1], reverse=True)[:15]]

labels, sources, targets, values = [], [], [], []
drug_idx, disease_idx, se_idx = {}, {}, {}

for drug in top_drugs:
    drug_idx[drug] = len(labels); labels.append(drug)
    for e in kg.edges:
        if e['source'] == drug:
            if e['relation'] == 'TREATS':
                if e['target'] not in disease_idx:
                    disease_idx[e['target']] = len(labels); labels.append(e['target'])
                sources.append(drug_idx[drug]); targets.append(disease_idx[e['target']]); values.append(1)
            elif e['relation'] == 'HAS_SIDE_EFFECT':
                if e['target'] not in se_idx:
                    se_idx[e['target']] = len(labels); labels.append(e['target'])
                sources.append(drug_idx[drug]); targets.append(se_idx[e['target']]); values.append(1)

fig = go.Figure(data=[go.Sankey(node=dict(pad=15, thickness=20, label=labels),
                                 link=dict(source=sources, target=targets, value=values))])
fig.update_layout(title='Drug → Disease → Side Effect Flow (Repository 1)', font=dict(size=10), width=1000, height=700)
fig.show()

### BLOCK 10: HEATMAP

In [ ]:
# Cell 14: Heatmap of drug-disease associations
drug_counts = Counter([e['source'] for e in kg.edges if e['relation'] == 'TREATS'])
disease_counts = Counter([e['target'] for e in kg.edges if e['relation'] == 'TREATS'])
top_drugs = [d for d, c in drug_counts.most_common(20)]
top_diseases = [d for d, c in disease_counts.most_common(15)]

matrix = np.zeros((len(top_drugs), len(top_diseases)))
for i, drug in enumerate(top_drugs):
    for j, disease in enumerate(top_diseases):
        if any(e['source'] == drug and e['target'] == disease and e['relation'] == 'TREATS'
               for e in kg.edges):
            matrix[i, j] = 1

fig = px.imshow(matrix, x=top_diseases, y=top_drugs, color_continuous_scale='Blues',
                title='Drug-Disease Association Matrix (Repository 1)', aspect='auto')
fig.update_layout(xaxis=dict(tickangle=45), height=500, width=900)
fig.show()

### BLOCK 11: EGO NETWORK

In [ ]:
# Cell 15: Ego network for a specific drug
top_drug = max(drug_degrees, key=lambda x: x[1])[0]
print(f"Selected drug: {top_drug}")

ego_nodes = set([top_drug])
for e in kg.edges:
    if e['source'] == top_drug:
        ego_nodes.add(e['target'])
    elif e['target'] == top_drug:
        ego_nodes.add(e['source'])

G_ego = G.subgraph(ego_nodes)
pos = nx.spring_layout(G_ego, k=1.5, seed=42)

fig, ax = plt.subplots(figsize=(14, 12))
colors = {'TREATS': 'green', 'HAS_SIDE_EFFECT': 'red', 'HAS_CLASS': 'blue',
          'MANUFACTURED_BY': 'orange', 'CONTRAINDICATED_FOR': 'purple',
          'CONTRAINDICATED_FROM_EXTERNAL': 'teal'}

for rel, color in colors.items():
    edges = [(u,v) for u,v,k,d in G_ego.edges(keys=True, data=True) if d.get('relation') == rel]
    if edges:
        nx.draw_networkx_edges(G_ego, pos, edgelist=edges, edge_color=color, width=2, alpha=0.7, ax=ax)

node_type_colors = {'Drug': 'lightcoral', 'Disease': 'lightgreen', 'SideEffect': 'gold',
                    'DrugClass': 'lightblue', 'Manufacturer': 'orange'}
node_colors = [node_type_colors.get(kg.nodes.get(n, {}).get('type', 'Unknown'), 'gray')
               for n in G_ego.nodes()]
nx.draw_networkx_nodes(G_ego, pos, node_color=node_colors, node_size=2000, alpha=0.9, ax=ax)
nx.draw_networkx_labels(G_ego, pos, {n: n[:20] for n in G_ego.nodes()}, font_size=8, ax=ax)

ax.set_title(f'Ego Network: {top_drug[:30]} (Includes external contraindications)', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

### BLOCK 12: EXPORT TO CSV

In [ ]:
# Cell 16: Export nodes and edges to CSV for Gephi
nodes_data = []
for nid, data in kg.nodes.items():
    nodes_data.append({'Id': nid, 'Label': nid, 'Type': data['type'],
                      **{k: str(v)[:100] for k, v in data['attributes'].items()}})
nodes_df = pd.DataFrame(nodes_data)
nodes_df.to_csv('kg_nodes.csv', index=False)
print(f"✓ Saved {len(nodes_df)} nodes to kg_nodes.csv")

edges_data = []
for e in kg.edges:
    edges_data.append({'Source': e['source'], 'Target': e['target'],
                      'Type': 'Directed', 'Label': e['relation'], 'Weight': 1})
edges_df = pd.DataFrame(edges_data)
edges_df.to_csv('kg_edges.csv', index=False)
print(f"✓ Saved {len(edges_df)} edges to kg_edges.csv")
print("\nFiles ready for import into Gephi!")

### BLOCK 13: FINAL SUMMARY WITH TWO REPOSITORIES

In [ ]:
# Cell 17: Final summary statistics with both repositories
print("=" * 70)
print("FINAL KNOWLEDGE GRAPH - TWO REPOSITORY INTEGRATION")
print("=" * 70)

print("\n📦 DATA REPOSITORIES USED:")
print("   ┌─────────────────────────────────────────────────────────────────┐")
print("   │ REPOSITORY 1: pharma_1.csv                                      │")
print("   │   - Type: Pharmaceutical drug database                         │")
print("   │   - Content: Drug names, classes, indications, side effects    │")
print("   └─────────────────────────────────────────────────────────────────┘")
print("   ┌─────────────────────────────────────────────────────────────────┐")
print("   │ REPOSITORY 2: Diseases_Symptoms.csv                             │")
print("   │   - Type: Medical symptom-disease ontology                     │")
print("   │   - Content: Disease names and their associated symptoms       │")
print("   └─────────────────────────────────────────────────────────────────┘")

print("\n📈 KNOWLEDGE GRAPH STATISTICS:")
print(f"   Total Drugs: {len(kg.get_nodes_by_type('Drug'))}")
print(f"   Total Diseases: {len(kg.get_nodes_by_type('Disease'))}")
print(f"   Total Side Effects: {len(kg.get_nodes_by_type('SideEffect'))}")
print(f"   Total Drug Classes: {len(kg.get_nodes_by_type('DrugClass'))}")
print(f"   Total Manufacturers: {len(kg.get_nodes_by_type('Manufacturer'))}")

print(f"\n🔗 TOTAL EDGES: {len(kg.edges)}")
edge_counts = {}
for e in kg.edges:
    edge_counts[e['relation']] = edge_counts.get(e['relation'], 0) + 1
for rel, count in sorted(edge_counts.items()):
    print(f"   {rel}: {count}")

print("\n" + "=" * 70)
print("✅ TWO DATA REPOSITORIES SUCCESSFULLY INTEGRATED!")
print("=" * 70)

### BLOCK 14: EXPORT TO OWL FORMAT

In [ ]:
# Cell 18: EXPORT TO OWL FORMAT (ONTOLOGY WEB LANGUAGE) - FIXED
!pip install owlready2 -q

from owlready2 import *
import warnings
import re
warnings.filterwarnings('ignore')

print("=" * 60)
print("EXPORTING KNOWLEDGE GRAPH TO OWL FORMAT")
print("=" * 60)

# Helper function to create safe names (handles special characters)
def make_safe_name(name):
    # Replace spaces and special chars with underscores
    safe = re.sub(r'[^a-zA-Z0-9_]', '_', str(name))
    # Remove multiple underscores
    safe = re.sub(r'_+', '_', safe)
    # Remove leading/trailing underscores
    safe = safe.strip('_')
    # Add prefix if starts with number
    if safe and safe[0].isdigit():
        safe = 'n_' + safe
    return safe

# Create a dictionary to store all individuals for lookup
individuals = {}

# Create a new ontology
onto = get_ontology("http://example.org/drug_disease_kg.owl")

with onto:
    # Define classes (node types)
    class Drug(Thing): pass
    class Disease(Thing): pass
    class SideEffect(Thing): pass
    class DrugClass(Thing): pass
    class Manufacturer(Thing): pass

    # Define object properties (relationships)
    class treats(Drug >> Disease): pass
    class has_side_effect(Drug >> SideEffect): pass
    class has_class(Drug >> DrugClass): pass
    class manufactured_by(Drug >> Manufacturer): pass
    class contraindicated_for(Drug >> Disease): pass

print("✓ Classes and properties defined")

# FIRST PASS: Create all individuals and store them in a dictionary
print("\n📦 Creating individuals...")

with onto:
    # Add Drug nodes
    drug_count = 0
    for drug in kg.get_nodes_by_type('Drug'):
        safe_name = make_safe_name(drug)
        ind = Drug(safe_name)
        individuals[drug] = ind  # Store original name -> individual
        drug_count += 1
        if drug_count % 1000 == 0:
            print(f"  Added {drug_count} drugs...")
    print(f"✓ Added {drug_count} Drug individuals")

    # Add Disease nodes
    disease_count = 0
    for disease in kg.get_nodes_by_type('Disease'):
        safe_name = make_safe_name(disease)
        ind = Disease(safe_name)
        individuals[disease] = ind
        disease_count += 1
    print(f"✓ Added {disease_count} Disease individuals")

    # Add SideEffect nodes
    se_count = 0
    for se in kg.get_nodes_by_type('SideEffect'):
        safe_name = make_safe_name(se)
        ind = SideEffect(safe_name)
        individuals[se] = ind
        se_count += 1
    print(f"✓ Added {se_count} SideEffect individuals")

    # Add DrugClass nodes
    dc_count = 0
    for dc in kg.get_nodes_by_type('DrugClass'):
        safe_name = make_safe_name(dc)
        ind = DrugClass(safe_name)
        individuals[dc] = ind
        dc_count += 1
    print(f"✓ Added {dc_count} DrugClass individuals")

    # Add Manufacturer nodes
    mfr_count = 0
    for mfr in kg.get_nodes_by_type('Manufacturer'):
        safe_name = make_safe_name(mfr)
        ind = Manufacturer(safe_name)
        individuals[mfr] = ind
        mfr_count += 1
    print(f"✓ Added {mfr_count} Manufacturer individuals")

# SECOND PASS: Add all relationships
print("\n🔗 Adding relationships...")

with onto:
    rel_count = 0
    error_count = 0

    for e in kg.edges:
        source_name = e['source']
        target_name = e['target']
        rel_type = e['relation']

        # Look up individuals
        source_ind = individuals.get(source_name)
        target_ind = individuals.get(target_name)

        # Skip if either doesn't exist
        if source_ind is None or target_ind is None:
            error_count += 1
            continue

        try:
            if rel_type == 'TREATS':
                source_ind.treats.append(target_ind)
            elif rel_type == 'HAS_SIDE_EFFECT':
                source_ind.has_side_effect.append(target_ind)
            elif rel_type == 'HAS_CLASS':
                source_ind.has_class.append(target_ind)
            elif rel_type == 'MANUFACTURED_BY':
                source_ind.manufactured_by.append(target_ind)
            elif 'CONTRAINDICATED' in rel_type:
                source_ind.contraindicated_for.append(target_ind)

            rel_count += 1
            if rel_count % 10000 == 0:
                print(f"  Added {rel_count} relationships...")
        except Exception as ex:
            error_count += 1
            continue

    print(f"✓ Added {rel_count} relationships")
    if error_count > 0:
        print(f"⚠️ Skipped {error_count} relationships due to lookup errors")

# Save the ontology
onto.save(file="drug_disease_knowledge_graph.owl", format="rdfxml")

print("\n" + "=" * 60)
print("✅ OWL FILE SAVED: drug_disease_knowledge_graph.owl")
print("=" * 60)

# Print file size
import os
if os.path.exists("drug_disease_knowledge_graph.owl"):
    size = os.path.getsize("drug_disease_knowledge_graph.owl") / (1024 * 1024)
    print(f"\n📁 File size: {size:.2f} MB")
    print("\nYou can now open this file in Protégé or other ontology editors!")